# 🧤 BdSL Sensor Glove Data Analysis

Analyse and classify gestures from the BdSL Sensor Glove dataset.

**Contents:**
1. Load sensor data
2. Visualise gestures
3. Feature engineering
4. Compare classifiers (Random Forest vs LSTM)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from typing import Tuple

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load the Dataset

Download first:
```bash
python scripts/download_datasets.py --dataset bdsl-sensor-glove
```

In [ ]:
from scripts.data_loader import BdSLSensorGloveDataset

train_ds = BdSLSensorGloveDataset(split='train')
val_ds   = BdSLSensorGloveDataset(split='val')
test_ds  = BdSLSensorGloveDataset(split='test')

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'Classes: {len(train_ds.GESTURE_TO_IDX)}')
print(f'Sensor shape per sample: {train_ds[0]["sensors"].shape}')  # (max_len, 11)

## 2. Visualise Different Gestures

Each sample has 11 sensor channels:
- 5 flex sensors (thumb, index, middle, ring, pinky)
- 3 accelerometer (x, y, z)
- 3 gyroscope (x, y, z)

In [ ]:
channel_names = [
    'Flex Thumb', 'Flex Index', 'Flex Middle', 'Flex Ring', 'Flex Pinky',
    'Accel X', 'Accel Y', 'Accel Z',
    'Gyro X', 'Gyro Y', 'Gyro Z'
]

target_gestures = ['KA', 'HELLO', 'WATER']
fig, axes = plt.subplots(len(target_gestures), 1, figsize=(14, 3 * len(target_gestures)))
if len(target_gestures) == 1:
    axes = [axes]

for row, gid in enumerate(target_gestures):
    for i, s in enumerate(train_ds.data):
        if s['gesture_id'] == gid:
            break
    sample = train_ds[i]
    sensors = sample['sensors'].numpy()
    mask = sample['mask'].numpy()
    t = int(mask.sum())
    for ch in range(11):
        axes[row].plot(sensors[:t, ch], label=channel_names[ch], alpha=0.8)
    axes[row].set_title(f'{gid} — {train_ds.GESTURE_NAMES[gid]}', fontsize=12)
    axes[row].set_xlabel('Time step')
    axes[row].legend(ncol=4, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
gesture_counts = Counter(s['gesture_id'] for s in train_ds.data)
gestures = sorted(gesture_counts.keys(), key=lambda g: gesture_counts[g], reverse=True)
counts = [gesture_counts[g] for g in gestures]

plt.figure(figsize=(14, 5))
sns.barplot(x=gestures, y=counts)
plt.xticks(rotation=45, ha='right')
plt.title('Gesture Distribution (Train Set)')
plt.ylabel('Count')
plt.show()

## 3. Feature Engineering

Extract statistical features from each time-series for traditional ML.

In [ ]:
def extract_features(dataset: BdSLSensorGloveDataset) -> Tuple[np.ndarray, np.ndarray]:
    """Extract per-channel statistical features from sensor data.
    Returns: X of shape (N, 55), y of shape (N,)
    """
    X, y = [], []
    for sample in dataset:
        sensors = sample['sensors'].numpy()      # (max_len, 11)
        mask = sample['mask'].numpy()              # (max_len,)
        t = int(mask.sum())
        seq = sensors[:t]  # actual readings

        feats = [
            seq.mean(axis=0),
            seq.std(axis=0) + 1e-8,
            seq.min(axis=0),
            seq.max(axis=0),
            (seq.max(axis=0) - seq.min(axis=0)),
        ]
        X.append(np.concatenate(feats))  # 11 channels * 5 stats = 55
        y.append(sample['label'].item())

    return np.array(X), np.array(y)

X_train, y_train = extract_features(train_ds)
X_val, y_val     = extract_features(val_ds)
X_test, y_test   = extract_features(test_ds)
print(f'Feature shape: {X_train.shape}')

## 4. Classifier Comparison

### 4a. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_acc = rf.score(X_test, y_test)
rf_preds = rf.predict(X_test)
print(f'Random Forest Test Accuracy: {rf_acc:.4f}')
print(classification_report(y_test, rf_preds, zero_division=0))

### 4b. LSTM

In [ ]:
class GestureLSTM(nn.Module):
    """LSTM for sequence-based gesture classification."""

    def __init__(self, input_size: int = 11, hidden_size: int = 128,
                 num_classes: int = 36, num_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, mask=None):
        # x: (batch, seq_len, 11)
        out, _ = self.lstm(x)
        # Use last valid time step (masked)
        if mask is not None:
            lengths = mask.sum(dim=1).long() - 1  # (batch,)
            idx = lengths.unsqueeze(1).unsqueeze(2).expand(-1, 1, out.size(2))
            out = out.gather(1, idx).squeeze(1)
        else:
            out = out[:, -1, :]
        return self.fc(out)

num_classes = len(train_ds.GESTURE_TO_IDX)
lstm_model = GestureLSTM(num_classes=num_classes).to(device)
print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

In [ ]:
from scripts.data_loader import collate_fn

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=32, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

epochs = 20
for epoch in range(1, epochs + 1):
    lstm_model.train()
    for batch in train_loader:
        sensors = batch['sensors'].to(device)
        masks   = batch['mask'].to(device)
        labels  = batch['labels'].to(device)
        optimizer.zero_grad()
        logits = lstm_model(sensors, masks)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

    # Validate
    lstm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            sensors = batch['sensors'].to(device)
            masks   = batch['mask'].to(device)
            labels  = batch['labels'].to(device)
            preds = lstm_model(sensors, masks).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}  val_acc={correct/total:.4f}')

In [ ]:
# Final test evaluation
lstm_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        sensors = batch['sensors'].to(device)
        masks   = batch['mask'].to(device)
        labels  = batch['labels'].to(device)
        preds = lstm_model(sensors, masks).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

lstm_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f'\nLSTM Test Accuracy: {lstm_acc:.4f}')

## 5. Comparison Summary

In [ ]:
results = {'Random Forest': rf_acc, 'LSTM': lstm_acc}
print('=== Classifier Comparison ===')
for name, acc in results.items():
    print(f'{name:20s}: {acc:.4f}')

plt.figure(figsize=(6, 4))
plt.bar(results.keys(), results.values(), color=['#4CAF50', '#2196F3'])
plt.ylabel('Test Accuracy')
plt.title('Classifier Comparison')
plt.ylim(0, 1)
for i, v in enumerate(results.values()):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.show()